# 🚀 Vision Engine — RT-DETRv11 Training
**Training a fast, local Object Detector for Bubbles & Narration**

### Steps
1. Mount Google Drive
2. Upload your `vision_yolo_dataset.zip` to Drive first
3. Run this notebook to train RT-DETRv11 on the free T4 GPU
4. Download the trained weights `best.pt`

**Runtime:** ~15-30 minutes for 50 epochs

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Install Ultralytics YOLOv11 ─────────────────────────────────────
!pip install -q ultralytics
import ultralytics
ultralytics.checks()
print('Ultralytics installed.')

In [ ]:
# ── Cell 3: Extract Dataset ─────────────────────────────────────────────────
import os
import zipfile
import shutil

DRIVE_ZIP_PATH = '/content/drive/MyDrive/vision_yolo_dataset.zip'
EXTRACT_DIR = '/content'
YOLO_DIR = '/content/yolo_format'

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

if os.path.exists(DRIVE_ZIP_PATH):
    print(f'Extracting {DRIVE_ZIP_PATH} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction complete.')
else:
    print(f'[ERROR] Zip not found at {DRIVE_ZIP_PATH}')

# Verify dataset.yaml
yaml_path = os.path.join(YOLO_DIR, 'dataset.yaml')
if os.path.exists(yaml_path):
    print('dataset.yaml found!')
    with open(yaml_path, 'r') as f:
        print(f.read())
else:
    print('[ERROR] dataset.yaml missing!')


In [ ]:
# ── Cell 4: Train RT-DETR Large ──────────────────────────────────────────────
from ultralytics import RTDETR

# We use YOLO11 Nano (rtdetr-l.pt) for extreme speed on CPU during local inference
model = RTDETR('rtdetr-l.pt')

results = model.train(
    data='/content/yolo_format/dataset.yaml',
    epochs=50,
    imgsz=1024,      # Large image size for manga readability
    batch=8,        # Maximize T4 GPU memory usage
    patience=10,     # Early stopping if no improvement
    name='vision_rtdetr_baseline',
    project='/content/runs'
)

print('Training complete!')

In [ ]:
# ── Cell 5: Evaluate on Test Set ────────────────────────────────────────────
# Runs validation on the unseen test split
metrics = model.val(data='/content/yolo_format/dataset.yaml', split='test')
print(f'Test mAP50: {metrics.box.map50:.3f}')

In [ ]:
# ── Cell 6: Copy Weights to Drive ───────────────────────────────────────────
import shutil
import os

best_weights = '/content/runs/vision_rtdetr_baseline/weights/best.pt'
drive_dest = '/content/drive/MyDrive/vision_rtdetr_baseline.pt'

if os.path.exists(best_weights):
    shutil.copy(best_weights, drive_dest)
    print(f'\n✅ SUCCESS! Weights saved to: {drive_dest}')
    print('👉 Now go to Google Drive, download vision_rtdetr_baseline.pt')
    print('👉 Move it to D:\\VISION\\weights\\ on your local machine.')
else:
    print('[ERROR] Could not find trained weights!')